In [ ]:
!git clone https://github.com/sukhrobnurali/tooltuned-qwen.git
%cd tooltuned-qwen
!pip install -e ".[colab]" --quiet
!pip uninstall -y torchcodec --quiet

In [ ]:
import sys, os
sys.path.insert(0, "/content/tooltuned-qwen/src")
# Colab exposes user secrets via google.colab.userdata, NOT os.environ --
# the Notebook-access toggle gates userdata.get, but env vars stay empty.
from google.colab import userdata
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    val = userdata.get(key)
    assert val, f"Set {key} in Colab secrets and toggle Notebook access on"
    os.environ[key] = val
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"])
print("ready")

In [ ]:
from tooltuned_qwen.training.train import train
adapter_path = train(config_path="configs/default.yaml")
adapter_path

In [ ]:
from tooltuned_qwen.hub.push_adapter import push
push(
    adapter_path,
    repo="sukhrobnurali/tooltuned-qwen-3.5-4b",
    private=False,
    token=os.environ["HF_TOKEN"],
)

In [ ]:
# Phase 3.3 -- regenerate the model card with BFCL section as TBD; Phase 4 fills it.
from tooltuned_qwen.hub.model_card import generate_card
card_path = generate_card(
    bfcl_results=None,
    training_config_path="configs/default.yaml",
    out_path="MODEL_CARD.md",
)
from huggingface_hub import HfApi
HfApi(token=os.environ["HF_TOKEN"]).upload_file(
    path_or_fileobj=card_path,
    path_in_repo="README.md",
    repo_id="sukhrobnurali/tooltuned-qwen-3.5-4b",
    repo_type="model",
)